# Download Qwen3 Embedding Model

This notebook downloads `Qwen/Qwen3-Embedding-0.6B` into `ABSA/embedding_model/qwen3_embedding_0_6b`.

Use this model for few-shot retrieval and example selection. Model weights are ignored by git.

## 1. Install the downloader

This installs only the Hugging Face Hub client needed to download the model files. The inference environment can be installed separately on Vast.

In [ ]:
%pip install -U huggingface_hub

## 2. Resolve paths

The code below finds the ABSA folder and sets the local embedding model directory.

In [ ]:
from pathlib import Path

MODEL_ID = "Qwen/Qwen3-Embedding-0.6B"

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, *cwd.parents]
ABSA_DIR = next(path for path in candidates if (path / "dataset" / "train.json").exists())
EMBEDDING_MODEL_DIR = ABSA_DIR / "embedding_model" / "qwen3_embedding_0_6b"
EMBEDDING_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("ABSA_DIR:", ABSA_DIR)
print("EMBEDDING_MODEL_DIR:", EMBEDDING_MODEL_DIR)

## 3. Optional login

`Qwen/Qwen3-Embedding-0.6B` is public. If Hugging Face asks for authentication or you hit rate limits, uncomment and run this cell.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 4. Download the embedding model

Run this on the machine where retrieval will be built. It can be the local machine or the Vast instance.

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=EMBEDDING_MODEL_DIR,
)

print("Downloaded", MODEL_ID, "to", EMBEDDING_MODEL_DIR)

## 5. Verify files

This checks that the local folder has the minimum files expected for a Hugging Face model snapshot.

In [ ]:
required_files = ["config.json", "tokenizer_config.json", "tokenizer.json"]
missing = [name for name in required_files if not (EMBEDDING_MODEL_DIR / name).exists()]
assert not missing, f"Missing files: {missing}"

print("Embedding model folder looks valid.")
for path in sorted(EMBEDDING_MODEL_DIR.iterdir()):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{path.name:45s} {size_mb:8.1f} MB")

## 6. Optional tokenizer sanity check

This does not load the full embedding model weights. It only verifies that the tokenizer can be loaded from the local folder.

In [ ]:
%pip install -U transformers safetensors

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_DIR, trust_remote_code=True)
sample = tokenizer("Represent this restaurant review for retrieving similar ABSA examples: great food and slow service")

print("Tokenizer:", tokenizer.__class__.__name__)
print("Sample token count:", len(sample["input_ids"]))

## 7. Expected use

The future few-shot retrieval code should load embeddings from:

```python
EMBEDDING_MODEL_DIR = ABSA_DIR / "embedding_model" / "qwen3_embedding_0_6b"
```

The generator model remains in:

```python
MODEL_DIR = ABSA_DIR / "model"
```